# TUGAS PRAKTIKUM 3 KECERDASAN KOMPUTASIONAL

| Informasi Mahasiswa | |
| :--- | :--- |
| **Nama** | Catherine Aprilia Harlina |
| **NRP** | 5054251012 |
| **Program Studi** | Rekayasa Kecerdasan Artifisial |
| **Mata Kuliah** | Kecerdasan Komputasional |

---

In [3]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules

def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None

ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )

if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings
import pandas as pd

warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

Environment : c:\Users\LEGION\Praktikum-KK\logics\praktikum\environment
Python      : 3.12.4
Check       : tt_entails(P & Q, Q) = True


## Soal 1 — Klasifikasi Syntax

### Analisis Ekspresi:

1. **`AdvisorOf(Ani)`**:
   - **Bentuk Utama**: **Term** (Function Application).
   - **Arity**: Function `AdvisorOf` ber-arity 1.
   - **Free Variable**: Tidak ada (`Ani` adalah constant).
   - **Alasan tidak bisa bernilai benar/salah**: `AdvisorOf(Ani)` menunjuk pada sebuah *object* (dosen wali dari Ani), bukan menyatakan klaim/proposisi. Sebuah object tidak memiliki truth value (True/False).

2. **`Lecturer(AdvisorOf(Ani))`**:
   - **Bentuk Utama**: **Formula Atomik**.
   - **Arity**: Predicate `Lecturer` ber-arity 1, Function `AdvisorOf` ber-arity 1.
   - **Free Variable**: Tidak ada.
   - **Status**: **Sentence** (karena tidak memuat free variable).

3. **`Takes(x, AI)`**:
   - **Bentuk Utama**: **Formula Atomik**.
   - **Arity**: Predicate `Takes` ber-arity 2.
   - **Free Variable**: `x` (karena diawali huruf kecil sesuai konvensi AIMA/logic.py).
   - **Status**: **Open Formula** (karena memuat free variable `x` yang belum terikat quantifier).

In [4]:
# Kode Verifikasi Soal 1
expr1 = expr('AdvisorOf(Ani)')
expr2 = expr('Lecturer(AdvisorOf(Ani))')
expr3 = expr('Takes(x, AI)')

expressions = [
    ("AdvisorOf(Ani)", expr1),
    ("Lecturer(AdvisorOf(Ani))", expr2),
    ("Takes(x, AI)", expr3),
]

for name, e in expressions:
    print(f"Expression : {name}")
    print(f"  Operator : {e.op}")
    print(f"  Args     : {e.args}")
    print(f"  Free Vars: {variables(e)}")
    print("-" * 40)

Expression : AdvisorOf(Ani)
  Operator : AdvisorOf
  Args     : (Ani,)
  Free Vars: set()
----------------------------------------
Expression : Lecturer(AdvisorOf(Ani))
  Operator : Lecturer
  Args     : (AdvisorOf(Ani),)
  Free Vars: set()
----------------------------------------
Expression : Takes(x, AI)
  Operator : Takes
  Args     : (x, AI)
  Free Vars: {x}
----------------------------------------


In [5]:
# Interpretation Model
D = {'Ani', 'Budi', 'Cici'}
Student = {'Ani', 'Budi'}
Diligent = {'Budi', 'Cici'}

# 1. Evaluasi Universal Statement: ∀x (Student(x) ⇒ Diligent(x))
# Formula: "Semua mahasiswa rajin"
all_student_diligent = all(
    (x not in Student) or (x in Diligent) for x in D
)
counterexamples = [x for x in D if (x in Student) and (x not in Diligent)]

# 2. Evaluasi Existential Statement: ∃x (Student(x) ∧ Diligent(x))
# Formula: "Ada mahasiswa yang rajin"
some_student_diligent = any(
    (x in Student) and (x in Diligent) for x in D
)
witnesses = [x for x in D if (x in Student) and (x in Diligent)]

print("1. ∀x (Student(x) ⇒ Diligent(x))")
print("   Hasil Evaluasi :", all_student_diligent)
if not all_student_diligent:
    print("   Counterexample :", counterexamples, "(Ani adalah mahasiswa, tetapi tidak rajin)")

print("\n2. ∃x (Student(x) ∧ Diligent(x))")
print("   Hasil Evaluasi :", some_student_diligent)
if some_student_diligent:
    print("   Witness        :", witnesses, "(Budi adalah mahasiswa dan sekaligus rajin)")

1. ∀x (Student(x) ⇒ Diligent(x))
   Hasil Evaluasi : False
   Counterexample : ['Ani'] (Ani adalah mahasiswa, tetapi tidak rajin)

2. ∃x (Student(x) ∧ Diligent(x))
   Hasil Evaluasi : True
   Witness        : ['Budi'] (Budi adalah mahasiswa dan sekaligus rajin)


## Soal 3 — Memperbaiki Quantifier

### 1. Analisis `∀x (Student(x) ∧ Smart(x))`
- **Makna Salah**: Formula ini menyatakan bahwa *setiap object di domain* adalah mahasiswa DAN sekaligus pintar.
- **Masalah**: Jika domain memuat object selain mahasiswa (misal `Kursi`), syarat `Student(Kursi)` bernilai `False`, sehingga seluruh formula bernilai `False` secara tidak tepat.
- **Formula yang Benar**: $\forall x \, (\text{Student}(x) \Rightarrow \text{Smart}(x))$

### 2. Analisis `∃x (Student(x) ⇒ Smart(x))`
- **Makna Salah**: Implikasi $P \Rightarrow Q$ bernilai `True` jika premis $P$ bernilai `False` (*vacuously true*).
- **Masalah**: Jika ada object di domain yang *bukan mahasiswa* (misal `Kursi`), maka `Student(Kursi)` bernilai `False`. Akibatnya `Student(Kursi) ⇒ Smart(Kursi)` bernilai `True`, sehingga klaim keberadaan (*existential*) menjadi terpenuhi secara keliru (*vacuous witness*).
- **Formula yang Benar**: $\exists x \, (\text{Student}(x) \land \text{Smart}(x))$

In [6]:
# Model Domain Kecil
domain_test = {'Ani', 'Kursi'}
student_test = {'Ani'}
smart_test = set()  # Tidak ada object yang pintar

# Formula Salah 1: Universal memakai Konjungsi (&)
wrong_univ = all((x in student_test) and (x in smart_test) for x in domain_test)
correct_univ = all((x not in student_test) or (x in smart_test) for x in domain_test)

# Formula Salah 2: Eksistensial memakai Implikasi (==>)
wrong_exist = any((x not in student_test) or (x in smart_test) for x in domain_test)
correct_exist = any((x in student_test) and (x in smart_test) for x in domain_test)

print("--- PENGUJIAN PADA DOMAIN {'Ani', 'Kursi'} (Tanpa Orang Pintar) ---")
print("Universal Salah (∀x Student(x) ∧ Smart(x))  :", wrong_univ)
print("Universal Benar (∀x Student(x) ⇒ Smart(x))  :", correct_univ)
print()
print("Existential Salah (∃x Student(x) ⇒ Smart(x)):", wrong_exist, "--> (Vacuously True karena 'Kursi' bukan Student)")
print("Existential Benar (∃x Student(x) ∧ Smart(x)):", correct_exist)

--- PENGUJIAN PADA DOMAIN {'Ani', 'Kursi'} (Tanpa Orang Pintar) ---
Universal Salah (∀x Student(x) ∧ Smart(x))  : False
Universal Benar (∀x Student(x) ⇒ Smart(x))  : False

Existential Salah (∃x Student(x) ⇒ Smart(x)): True --> (Vacuously True karena 'Kursi' bukan Student)
Existential Benar (∃x Student(x) ∧ Smart(x)): False


## Soal 4 — Scope dan Urutan Quantifier

### 1. "Setiap dosen mengajar sedikitnya satu mahasiswa."
- **FOL Formal**: $\forall x \, (\text{Lecturer}(x) \Rightarrow \exists y \, (\text{Student}(y) \land \text{Teaches}(x, y)))$
- **Perbedaan Scope & Urutan**: Quantifier $\forall x$ berada di luar $\exists y$. Mahasiswa ($y$) yang diajar dapat berbeda-beda untuk setiap dosen ($x$).

### 2. "Ada satu mahasiswa yang diajar oleh setiap dosen."
- **FOL Formal**: $\exists y \, (\text{Student}(y) \land \forall x \, (\text{Lecturer}(x) \Rightarrow \text{Teaches}(x, y)))$
- **Perbedaan Scope & Urutan**: Quantifier $\exists y$ berada di luar $\forall x$. Berarti ada **satu mahasiswa spesifik** ($y$) yang sama, yang menjadi murid dari seluruh dosen ($x$).

In [7]:
# Model Domain
lecturers = {'PakBudi', 'BuRina'}
students = {'Ani', 'Cici'}

# Pengajaran: PakBudi mengajar Ani, BuRina mengajar Cici
teaches_rel = {('PakBudi', 'Ani'), ('BuRina', 'Cici')}

# Kalimat 1: ∀x (Lecturer(x) ⇒ ∃y (Student(y) ∧ Teaches(x,y)))
cond1 = all(
    any((d, m) in teaches_rel for m in students)
    for d in lecturers
)

# Kalimat 2: ∃y (Student(y) ∧ ∀x (Lecturer(x) ⇒ Teaches(x,y)))
cond2 = any(
    all((d, m) in teaches_rel for d in lecturers)
    for m in students
)

print("1. ∀x ∃y (Setiap dosen mengajar minimal 1 mahasiswa) :", cond1)
print("2. ∃y ∀x (Ada 1 mahasiswa yang diajar SELUA dosen)    :", cond2)

1. ∀x ∃y (Setiap dosen mengajar minimal 1 mahasiswa) : True
2. ∃y ∀x (Ada 1 mahasiswa yang diajar SELUA dosen)    : False


## Soal 5 — Translation & Encoding Executable

### Kalimat 1: "Semua mahasiswa yang mengambil AI dan lulus Pemrograman boleh mengikuti praktikum AI."
- **FOL Formal**: $\forall x \, ((\text{Student}(x) \land \text{Takes}(x, \text{AI}) \land \text{Passed}(x, \text{Prog})) \Rightarrow \text{Eligible}(x, \text{AILab}))$
- **Free Variables**: Tidak ada.
- **Bound Variables**: $\{x\}$.
- **FolKB Encoding**: `(Student(x) & Takes(x, AI) & Passed(x, Prog)) ==> Eligible(x, AILab)`

### Kalimat 2: "Ada mahasiswa yang mengambil AI."
- **FOL Formal**: $\exists x \, (\text{Student}(x) \land \text{Takes}(x, \text{AI}))$
- **Free Variables**: Tidak ada.
- **Bound Variables**: $\{x\}$.
- **Penjelasan Fresh Constant**: Karena `FolKB` tidak mendukung quantifier $\exists$ secara langsung, kita menggunakan *Skolemization* dengan memperkenalkan **fresh constant** (misalnya `S_AI`). Constant ini dinamakan *fresh* karena harus berupa simbol baru yang belum pernah dipakai di Knowledge Base (KB) untuk menghindari bentrokan makna dengan individu/object lain yang sudah ada.
- **FolKB Encoding**:
  - `Student(S_AI)`
  - `Takes(S_AI, AI)`

In [8]:
# 1. Definisikan Clause
rule_eligible = expr('(Student(x) & Takes(x, AI) & Passed(x, Prog)) ==> Eligible(x, AILab)')
fact_student = expr('Student(S_AI)')
fact_takes = expr('Takes(S_AI, AI)')

# 2. Construct FolKB
my_kb = FolKB([
    rule_eligible,
    fact_student,
    fact_takes
])

print("=== DAFTAR CLAUSE DALAM FOLKB ===")
for i, clause in enumerate(my_kb.clauses, 1):
    print(f"Clause {i}          : {clause}")
    print(f"Is Definite Clause? : {is_definite_clause(clause)}")
    print("-" * 55)

=== DAFTAR CLAUSE DALAM FOLKB ===
Clause 1          : (((Student(x) & Takes(x, AI)) & Passed(x, Prog)) ==> Eligible(x, AILab))
Is Definite Clause? : True
-------------------------------------------------------
Clause 2          : Student(S_AI)
Is Definite Clause? : True
-------------------------------------------------------
Clause 3          : Takes(S_AI, AI)
Is Definite Clause? : True
-------------------------------------------------------
